In [1]:
!pip install google-play-scraper vaderSentiment

import pandas as pd
from google_play_scraper import app, reviews, Sort
import time
from datetime import datetime
import os
import re
import json
import hashlib
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer

# ========================
# CONFIGURATION (Phase 0)
# ========================

APPS = {
    'Social Media': [
        {'name': 'WhatsApp', 'id': 'com.whatsapp'},
        {'name': 'Instagram', 'id': 'com.instagram.android'}
    ],
    'E-commerce': [
        {'name': 'Daraz', 'id': 'com.daraz.android'},
        {'name': 'Amazon', 'id': 'com.amazon.mShop.android.shopping'}
    ],
    'Food Delivery': [
        {'name': 'Foodpanda', 'id': 'com.global.foodpanda.android'},
        {'name': 'Careem', 'id': 'com.careem.acma'}
    ],
    'Banking': [
        {'name': 'JazzCash', 'id': 'com.techlogix.mobilinkcustomer'},
        {'name': 'Easypaisa', 'id': 'pk.com.telenor.phoenix'}
    ],
    'Entertainment': [
        {'name': 'Netflix', 'id': 'com.netflix.mediaclient'},
        {'name': 'Spotify', 'id': 'com.spotify.music'}
    ]
}

VERSION = "v1.0"
COLLECTION_TIMESTAMP = datetime.now().strftime("%Y%m%d_%H%M%S")
OUTPUT_DIR = f'data/raw_reviews_{VERSION}_{COLLECTION_TIMESTAMP}'

# Realistic rating distribution (positive bias like real Play Store)
RATING_TARGETS = {
    5: 500,
    4: 200,
    3: 100,
    2: 200,
    1: 500
}

MAX_WORDS_PER_REVIEW = 40

# ========================
# HELPER FUNCTIONS (Phase 1)
# ========================

def safe_get(dictionary, key, default=''):
    """Safely extract values"""
    try:
        return dictionary.get(key, default)
    except:
        return default

def generate_hash_id(text, timestamp, username):
    """Fallback unique ID generator"""
    combined = f"{text}_{timestamp}_{username}"
    return hashlib.md5(combined.encode()).hexdigest()[:16]

def count_urdu_words(text):
    """Count Roman Urdu words (ROADMAP: has_urdu_words flag)"""
    urdu_words = ['hai', 'nahi', 'acha', 'bohat', 'kya', 'app', 'bahut', 
                  'achha', 'zabardast', 'mast', 'bakwas', 'bekar', 'kharab',
                  'theek', 'sahi', 'galat', 'mazay', 'bewakoof']
    if not text:
        return 0
    text_lower = text.lower()
    return sum(1 for word in urdu_words if word in text_lower)

def check_generic_username(username):
    """ROADMAP: generic_username flag"""
    if not username:
        return 1
    pattern = r'^(user|reviewer|test|account|google|android|person|member)\d*$'
    return 1 if re.match(pattern, username.strip(), re.IGNORECASE) else 0

def count_promo_words(text):
    """ROADMAP: contains_promo_words flag"""
    promo_keywords = ['download', 'free', 'buy now', 'click here', 'visit', 
                     'limited time', 'offer', 'discount', 'deal', 'sale',
                     'cheap', 'install', 'link', 'website', 'promocode']
    if not text:
        return 0
    text_lower = text.lower()
    return sum(1 for keyword in promo_keywords if keyword in text_lower)

def is_valid_review(text, max_words=40):
    """Quality filter for scraping"""
    if not text or len(text.strip()) < 5:
        return False
    
    word_count = len(text.split())
    if word_count > max_words:
        return False
    
    url_count = text.count('http') + text.count('www.') + text.count('.com')
    if url_count > 2:
        return False
    
    return True

def compute_sentiment_score(text):
    """ROADMAP: sentiment_score using VADER"""
    analyzer = SentimentIntensityAnalyzer()
    if not text:
        return 0.0
    scores = analyzer.polarity_scores(text)
    return scores['compound']  # Range: -1 (negative) to +1 (positive)

def save_metadata(output_dir, config, stats):
    """ROADMAP Phase 0: Version control + metadata"""
    metadata = {
        'version': config['version'],
        'timestamp': config['timestamp'],
        'collection_date': datetime.now().isoformat(),
        'rating_targets': config['rating_targets'],
        'max_words': config['max_words'],
        'apps': config['apps'],
        'statistics': stats
    }
    
    metadata_file = f"{output_dir}/README.json"
    with open(metadata_file, 'w', encoding='utf-8') as f:
        json.dump(metadata, f, indent=2, ensure_ascii=False)
    
    print(f"   ✅ Metadata saved: {metadata_file}")

# ========================
# SCRAPING (Realistic Balance)
# ========================

def get_app_info(app_id):
    """Get app metadata"""
    try:
        info = app(app_id)
        return {
            'app_id': app_id,
            'app_name': safe_get(info, 'title'),
            'category': safe_get(info, 'genre'),
            'rating': safe_get(info, 'score', 0),
            'total_reviews': safe_get(info, 'reviews', 0),
            'installs': safe_get(info, 'installs'),
            'developer': safe_get(info, 'developer', 'Unknown')
        }
    except Exception as e:
        print(f"❌ Error getting app info: {e}")
        return None

def scrape_balanced_reviews(app_id, app_name):
    """Scrape with realistic distribution - max 50 per rating per iteration"""
    balanced_reviews = {1: [], 2: [], 3: [], 4: [], 5: []}
    
    print(f"\n🔍 Scraping {app_name}...")
    
    MAX_ITERATIONS = 25  # Total iterations limit
    MAX_PER_RATING_PER_ITERATION = 50  # Max 50 reviews per rating per iteration
    
    languages = ['en', 'ur']
    sort_methods = [Sort.NEWEST, Sort.MOST_RELEVANT, Sort.RATING]
    
    total_iterations = 0
    
    for lang in languages:
        for sort_method in sort_methods:
            continuation_token = None
            
            while total_iterations < MAX_ITERATIONS:
                try:
                    # Stop if all targets met
                    if all(len(balanced_reviews[r]) >= RATING_TARGETS[r] for r in [1,2,3,4,5]):
                        print(f"   ✅ All targets met!")
                        break
                    
                    result, continuation_token = reviews(
                        app_id,
                        lang=lang,
                        country='pk',
                        sort=sort_method,
                        count=200,
                        continuation_token=continuation_token
                    )
                    
                    if not result:
                        break
                    
                    # Track how many added per rating in THIS iteration
                    iteration_added = {1: 0, 2: 0, 3: 0, 4: 0, 5: 0}
                    
                    # Organize by rating with per-iteration limit
                    for review in result:
                        rating = safe_get(review, 'score', 0)
                        text = safe_get(review, 'content', '')
                        
                        if not is_valid_review(text, MAX_WORDS_PER_REVIEW):
                            continue
                        
                        # Check if we can add more for this rating
                        current_count = len(balanced_reviews[rating])
                        iteration_count = iteration_added[rating]
                        
                        # Add only if:
                        # 1. Haven't reached target for this rating
                        # 2. Haven't added 50 in this iteration
                        # 3. Not a duplicate
                        existing_texts = set(rv['content'] for rv in balanced_reviews[rating])
                        
                        if (current_count < RATING_TARGETS[rating] and 
                            iteration_count < MAX_PER_RATING_PER_ITERATION and
                            text not in existing_texts):
                            balanced_reviews[rating].append(review)
                            iteration_added[rating] += 1
                    
                    # Progress
                    counts = {r: len(revs) for r, revs in balanced_reviews.items()}
                    added_this_iter = sum(iteration_added.values())
                    print(f"   Iteration {total_iterations+1}/{MAX_ITERATIONS} → 1★:{counts[1]} 2★:{counts[2]} 3★:{counts[3]} 4★:{counts[4]} 5★:{counts[5]} | Added: {added_this_iter}")
                    
                    time.sleep(1)
                    total_iterations += 1
                    
                    # If no continuation token or added nothing, try next sort/lang
                    if not continuation_token or added_this_iter == 0:
                        break
                
                except Exception as e:
                    print(f"   ⚠️ Error: {e}")
                    time.sleep(3)
                    total_iterations += 1
                    continue
            
            # Stop if all targets met or max iterations reached
            if all(len(balanced_reviews[r]) >= RATING_TARGETS[r] for r in [1,2,3,4,5]):
                break
            if total_iterations >= MAX_ITERATIONS:
                print(f"   ⚠️ Reached max {MAX_ITERATIONS} iterations")
                break
        
        if total_iterations >= MAX_ITERATIONS:
            break
    
    # Combine all ratings (take whatever we got, up to target)
    final_reviews = []
    for rating in [1,2,3,4,5]:
        final_reviews.extend(balanced_reviews[rating][:RATING_TARGETS[rating]])
    
    print(f"   ✅ Collected {len(final_reviews)} reviews in {total_iterations} iterations")
    return final_reviews

# ========================
# PHASE 1: PREPROCESSING & FLAGGING
# ========================

def process_reviews_phase1(reviews_data, app_name, category):
    """ROADMAP Phase 1: Extract ALL flags without deleting"""
    processed = []
    
    for review in reviews_data:
        # Basic extraction
        review_id = safe_get(review, 'reviewId', '')
        username = safe_get(review, 'userName', 'Anonymous')
        timestamp = safe_get(review, 'at', '')
        text_raw = safe_get(review, 'content', '')
        device = safe_get(review, 'reviewDevice', 'Unknown')
        rating = safe_get(review, 'score', 0)
        thumbs_up = safe_get(review, 'thumbsUpCount', 0)
        
        # Fallback ID if missing
        if not review_id:
            review_id = generate_hash_id(text_raw, str(timestamp), username)
        
        # ROADMAP: Keep raw + cleaned text
        text_original = text_raw
        text_cleaned = text_raw.lower().strip() if text_raw else ''
        
        # ROADMAP Phase 1 FLAGS (compute but don't delete)
        
        # Text features
        token_count = len(text_cleaned.split())
        char_count = len(text_cleaned)
        num_emojis = len([c for c in text_cleaned if ord(c) > 127000])
        num_urls = text_cleaned.count('http') + text_cleaned.count('www.')
        num_uppercase_words = len([w for w in text_raw.split() if w.isupper() and len(w) > 1])
        
        words = text_cleaned.split()
        unique_word_ratio = len(set(words)) / max(len(words), 1)
        
        # ROADMAP: Promo + Urdu flags
        contains_promo_words = 1 if count_promo_words(text_cleaned) > 0 else 0
        has_urdu_words = 1 if count_urdu_words(text_cleaned) > 0 else 0
        
        # ROADMAP: Sentiment score
        sentiment_score = compute_sentiment_score(text_cleaned)
        
        # ROADMAP: Device flag
        device_missing = 1 if device == 'Unknown' or not device else 0
        
        # ROADMAP: Generic username flag
        generic_username = check_generic_username(username)
        
        processed.append({
            # ROADMAP Phase 0: Immutable raw data
            'review_id': review_id,
            'app_name': app_name,
            'category': category,
            'username': username,
            'device': device,
            'rating': rating,
            'timestamp': timestamp,
            'text_original': text_original,
            'text_cleaned': text_cleaned,
            'thumbs_up': thumbs_up,
            
            # ROADMAP Phase 1: Text flags
            'token_count': token_count,
            'char_count': char_count,
            'num_emojis': num_emojis,
            'num_urls': num_urls,
            'num_uppercase_words': num_uppercase_words,
            'unique_word_ratio': unique_word_ratio,
            'contains_promo_words': contains_promo_words,
            'has_urdu_words': has_urdu_words,
            
            # ROADMAP Phase 1: Sentiment
            'sentiment_score': sentiment_score,
            
            # ROADMAP Phase 1: Behavioral (computed later)
            'device_missing': device_missing,
            'generic_username': generic_username
        })
    
    df = pd.DataFrame(processed)
    
    # Parse timestamps
    df['timestamp'] = pd.to_datetime(df['timestamp'], errors='coerce')
    df = df.sort_values('timestamp').reset_index(drop=True)
    
    # ROADMAP Phase 1: Behavioral flags (user-level)
    
    # User total reviews
    user_counts = df['username'].value_counts().to_dict()
    df['user_total_reviews'] = df['username'].map(user_counts)
    
    # Time diff between reviews (same user)
    df['time_diff_seconds'] = df.groupby('username')['timestamp'].diff().dt.total_seconds()
    df['time_diff_seconds'] = df['time_diff_seconds'].fillna(0)
    
    # ROADMAP: Burst flag
    df['is_burst'] = ((df['time_diff_seconds'] > 0) & 
                      (df['time_diff_seconds'] < 300)).astype(int)
    
    # ROADMAP: Same text count (duplicates across users)
    text_counts = df['text_cleaned'].value_counts().to_dict()
    df['same_text_count'] = df['text_cleaned'].map(text_counts)
    
    # ROADMAP Phase 1: Rating-text mismatch
    # Negative sentiment (-0.5 or lower) with high rating (4-5) = mismatch
    # Positive sentiment (+0.5 or higher) with low rating (1-2) = mismatch
    df['rating_text_mismatch'] = (
        ((df['sentiment_score'] <= -0.5) & (df['rating'] >= 4)) |
        ((df['sentiment_score'] >= 0.5) & (df['rating'] <= 2))
    ).astype(int)
    
    # User rating patterns
    user_rating_mean = df.groupby('username')['rating'].mean().to_dict()
    user_rating_std = df.groupby('username')['rating'].std().fillna(0).to_dict()
    df['user_avg_rating'] = df['username'].map(user_rating_mean)
    df['user_rating_std'] = df['username'].map(user_rating_std)
    
    return df

# ========================
# MAIN COLLECTION
# ========================

def collect_all_data():
    """ROADMAP Phase 0 + Phase 1: Collect + Preprocess + Flag"""
    
    os.makedirs(OUTPUT_DIR, exist_ok=True)
    
    all_data = []
    metadata_list = []
    
    print("=" * 60)
    print(f"📱 PHASE 1: DATA COLLECTION + PREPROCESSING")
    print(f"📅 Version: {VERSION} | Timestamp: {COLLECTION_TIMESTAMP}")
    print("=" * 60)
    
    for category, apps in APPS.items():
        print(f"\n📂 Category: {category}")
        
        for app_dict in apps:
            app_name = app_dict['name']
            app_id = app_dict['id']
            
            # Get metadata
            app_info = get_app_info(app_id)
            if app_info:
                app_info['category_custom'] = category
                metadata_list.append(app_info)
            
            # Scrape
            reviews_data = scrape_balanced_reviews(app_id, app_name)
            
            if reviews_data:
                # ROADMAP Phase 1: Process with all flags
                df = process_reviews_phase1(reviews_data, app_name, category)
                all_data.append(df)
                
                # Save individual file
                filename = f"{OUTPUT_DIR}/{app_name.lower().replace(' ', '_')}_preprocessed.csv"
                df.to_csv(filename, index=False, encoding='utf-8-sig')
                print(f"   💾 Saved: {filename}")
            
            time.sleep(2)
    
    # Combine
    if all_data:
        combined_df = pd.concat(all_data, ignore_index=True)
        
        # Save combined
        combined_file = f"{OUTPUT_DIR}/preprocessed.csv"
        combined_df.to_csv(combined_file, index=False, encoding='utf-8-sig')
        
        # Save metadata
        metadata_df = pd.DataFrame(metadata_list)
        metadata_file = f"{OUTPUT_DIR}/apps_metadata.csv"
        metadata_df.to_csv(metadata_file, index=False)
        
        # ROADMAP Phase 0: Save version control metadata
        stats = {
            'total_reviews': len(combined_df),
            'total_apps': len(metadata_df),
            'rating_distribution': combined_df['rating'].value_counts().to_dict(),
            'flagged_stats': {
                'promo_words': int(combined_df['contains_promo_words'].sum()),
                'urdu_words': int(combined_df['has_urdu_words'].sum()),
                'bursts': int(combined_df['is_burst'].sum()),
                'duplicates': int((combined_df['same_text_count'] >= 3).sum()),
                'rating_mismatch': int(combined_df['rating_text_mismatch'].sum()),
                'generic_usernames': int(combined_df['generic_username'].sum())
            }
        }
        
        config = {
            'version': VERSION,
            'timestamp': COLLECTION_TIMESTAMP,
            'rating_targets': RATING_TARGETS,
            'max_words': MAX_WORDS_PER_REVIEW,
            'apps': APPS
        }
        
        save_metadata(OUTPUT_DIR, config, stats)
        
        # Report
        print("\n" + "=" * 60)
        print("✅ PHASE 1 COMPLETE: PREPROCESSING & FLAGGING")
        print("=" * 60)
        print(f"\n📊 Total Reviews: {len(combined_df)}")
        print(f"📱 Total Apps: {len(metadata_df)}")
        print(f"🏷️  Total Flags: {len([c for c in combined_df.columns if c.startswith('flag_') or c in ['is_burst', 'same_text_count', 'rating_text_mismatch']])}")
        
        print("\n🚩 FLAGGED STATISTICS:")
        print(f"   • Promo Words: {stats['flagged_stats']['promo_words']}")
        print(f"   • Urdu Words: {stats['flagged_stats']['urdu_words']}")
        print(f"   • Burst Posting: {stats['flagged_stats']['bursts']}")
        print(f"   • Duplicates (≥3 users): {stats['flagged_stats']['duplicates']}")
        print(f"   • Rating-Text Mismatch: {stats['flagged_stats']['rating_mismatch']}")
        print(f"   • Generic Usernames: {stats['flagged_stats']['generic_usernames']}")
        
        print("\n⭐ RATING DISTRIBUTION:")
        for rating, count in combined_df['rating'].value_counts().sort_index().items():
            pct = (count/len(combined_df))*100
            print(f"   {rating}★: {count:4d} ({pct:5.1f}%)")
        
        print(f"\n✅ Output Directory: {OUTPUT_DIR}")
        print(f"✅ Main File: preprocessed.csv")
        print(f"✅ Ready for Phase 2: Scoring Components\n")
        
        return combined_df, metadata_df
    
    return None, None

# ========================
# RUN
# ========================

if __name__ == "__main__":
    df, metadata = collect_all_data()

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.2/50.2 kB 1.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 126.0/126.0 kB 5.6 MB/s eta 0:00:00
📱 PHASE 1: DATA COLLECTION + PREPROCESSING
📅 Version: v1.0 | Timestamp: 20251110_190301

📂 Category: Social Media

🔍 Scraping WhatsApp...
   Iteration 1/25 → 1★:33 2★:8 3★:7 4★:11 5★:50 | Added: 109
   Iteration 2/25 → 1★:65 2★:14 3★:16 4★:21 5★:100 | Added: 107
   Iteration 3/25 → 1★:94 2★:24 3★:22 4★:35 5★:150 | Added: 109
   Iteration 4/25 → 1★:118 2★:25 3★:31 4★:46 5★:200 | Added: 95
   Iteration 5/25 → 1★:133 2★:29 3★:38 4★:59 5★:250 | Added: 89
   Iteration 6/25 → 1★:161 2★:32 3★:46 4★:69 5★:300 | Added: 99
   Iteration 7/25 → 1★:189 2★:37 3★:53 4★:83 5★:350 | Added: 104
   Iteration 8/25 → 1★:224 2★:43 3★:58 4★:91 5★:400 | Added: 104
   Iteration 9/25 → 1★:268 2★:47 3★:62 4★:102 5★:450 | Added: 113
   Iteration 10/25 → 1★:299 2★:55 3★:72 4★:105 5★:500 | Added: 102
   Iteration 11/25 → 1★:328 2★:58 3★:80 4★:119 5★:500 | Ad